In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv("05_retail_sales_practice.csv")

In [ ]:
df.head(3)

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
df[df.duplicated()]

In [ ]:
print("Before Drop Duplicate :",df.shape)

In [ ]:
df = df.drop_duplicates()

In [ ]:
print("After Drop Duplicate :",df.shape)

In [ ]:
print(df.index)

In [ ]:
(df.index == range(len(df))).all()

In [ ]:
df.index.min(),df.index.max()

In [ ]:
df.isnull().sum()

In [ ]:
sns.histplot(df['CustomerAge'], kde=True)

In [ ]:
sns.boxplot(data=df,x='CustomerAge')

# no outliers and bell-shaped, symmetric means use mean

In [ ]:
df.info()

# **fill NaN value to mean of customerage and change datatype float into int**

In [ ]:
df['CustomerAge'] = df['CustomerAge'].fillna(round(df['CustomerAge'].mean())).astype(int)

In [ ]:
df['CustomerAge'].isnull().sum()

In [ ]:
df.isnull().sum()

# **fill City NaN**

In [ ]:
df['City'].unique()

In [ ]:
df['City'].value_counts()

In [ ]:
plt.figure(figsize=(10,5))
sns.countplot(data=df, x='City', order=df['City'].value_counts().index)
plt.xticks(rotation=45)
plt.show()

In [ ]:
df['City'].mode()[0]

# **in city Hyderabad is most frequent so replace NaN with Hyderabad**

In [ ]:
df['City'] = df['City'].fillna('Hyderabad')

In [ ]:
df['City'].isnull().sum()

In [ ]:
df.isnull().sum()

# **Replace NaN from Rating**

In [ ]:
df['Rating'].isnull().sum()

In [ ]:
sns.boxplot(data=df,x='Rating')

In [ ]:
df['Rating'].mean()

In [ ]:
df['Rating'].median()

In [ ]:
df['Rating'].mode()[0]

In [ ]:
df['Rating'].value_counts()

# **Fill NaN in Rating with median**

In [ ]:
df['Rating'] = df['Rating'].fillna(df['Rating'].median())

In [ ]:
df['Rating'].isnull().sum()

In [ ]:
df.isnull().sum()

# **Detect outliers in Sales using the IQR method or Z-score (NumPy), and decide how to treat them**

In [ ]:
plt.figure(figsize=(10,5))
sns.boxplot(data = df, x ='Sales')
plt.show()

# Let's calculate the IQR (Interquartile Range) for Sales and flag the outliers.

In [ ]:
df['Sales'].describe()

In [ ]:
# Find IQR
Q1 = df['Sales'].quantile(0.25)
Q3 = df['Sales'].quantile(0.75)
IQR = Q3 - Q1
print(IQR)

In [ ]:
# Find lower bound And upper bound
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
print(lower_bound)
print(upper_bound)

In [ ]:
print('Q1: ',Q1)
print('Q3: ',Q3)
print('IQR: ',IQR)
print('lower_bound: ',lower_bound)
print('upper_bound: ',upper_bound)

In [ ]:
df.head(1)

In [ ]:
# find outliers
outliers = df[(df['Sales']< lower_bound) | (df['Sales'] > upper_bound)]
print(outliers.shape)


In [ ]:
outliers[['OrderID','Category','Quantity','UnitPrice','Sales','Discount(%)']]

In [ ]:
outliers['ExpectedSales'] = outliers['Quantity'] * outliers['UnitPrice'] * (1 - outliers['Discount(%)']/100)
outliers[['OrderID','Sales','ExpectedSales']]

In [ ]:
# fake outliers ko identify karo
fake_outliers = outliers[outliers['Sales'] > outliers['ExpectedSales'] * 5]
print(fake_outliers[['OrderID','Sales','ExpectedSales']])

In [ ]:
# unko ExpectedSales se replace karo
df.loc[fake_outliers.index, 'Sales'] = fake_outliers['ExpectedSales']

In [ ]:
df.loc[fake_outliers.index, ['OrderID','Sales']]

In [ ]:
# all outlier are clear let's check
Q1 = df['Sales'].quantile(0.25)
Q3 = df['Sales'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers_check = df[(df['Sales'] < lower_bound) | (df['Sales'] > upper_bound)]
print(outliers_check.shape)

In [ ]:
outliers_check['ExpectedSales'] = outliers_check['Quantity'] * outliers_check['UnitPrice'] * (1 - outliers_check['Discount(%)']/100)
outliers_check[['OrderID','Sales','ExpectedSales']]

In [ ]:
plt.figure(figsize=(10,5))
sns.boxplot(data = df, x ='Sales')
plt.show()

# **5. Convert OrderDate to a proper datetime type and extract Year, Month, and Weekday into new columns.**

In [ ]:
df.info()

In [ ]:
df.head(1)

In [ ]:
df['OrderDate'] = pd.to_datetime(df['OrderDate'])

In [ ]:
df['Date'] = df['OrderDate'].dt.day
df['Month'] = df['OrderDate'].dt.month
df['Year'] = df['OrderDate'].dt.year

In [ ]:
df.head(2)

# **6. Create a new column ProfitMargin = Profit / Sales.:**

In [ ]:
df['ProfitMargin'] = (df['Profit'] / df['Sales']) * 100

In [ ]:
df.head(2)

# **Part 2 — Exploratory Analysis (Pandas + NumPy)**

# 7. What are the total Sales and total Profit for the company overall?

In [ ]:
print("total sale :",df['Sales'].sum())
print("total profit :",df['Profit'].sum())

# 8. Which Category generates the highest revenue? Which generates the highest profit margin?

In [ ]:
df.head(1)

In [ ]:
df.groupby('Category')['Sales'].sum()

In [ ]:
df.groupby('Category')['ProfitMargin'].mean()

# 9. Which City has the most orders? Which city has the highest average order value?

In [ ]:
df.groupby('City')['OrderID'].count()

In [ ]:
# which City has the highest average order value?
df.groupby('City')['Sales'].mean()

# 10. Is there a relationship between Discount(%) and Profit? Do higher discounts hurt profit?

In [ ]:
df.groupby('Discount(%)')['Profit'].mean()

As Discount increases → Profit decreases (generally). The pattern is clear:

0% discount → highest average profit (₹3,834)
10% discount → profit drops significantly (₹2,739)
25% discount → profit stays low (₹3,250)

# 11. What is the average customer Rating by Category? Any category performing poorly?

In [ ]:
df.head(1)

In [ ]:
df.groupby('Category')['Rating'].mean()

# 12. Which PaymentMethod is most popular? Does it vary by city?

In [ ]:
df['PaymentMethod'].value_counts()

In [ ]:
df.groupby('City')['PaymentMethod'].value_counts().unstack()

In [ ]:
df.groupby(['City','PaymentMethod'])['OrderID'].count().unstack()

# 13. Use groupby + pivot_table to build a Category × Month sales summary.

In [ ]:
pd.pivot_table(df, values='Sales', index='Category', columns='Month', aggfunc='sum')

# 14. Segment customers into Age groups (e.g., 18–25, 26–35, 36–50, 50+) using pd.cut — which group spends the most?

In [ ]:
df['AgeGroup'] = pd.cut(df['CustomerAge'], bins=[18,25,35,50,65], labels=['18-25','26-35','36-50','50+'])
df.groupby('AgeGroup')['Sales'].mean()

In [ ]:
df.head()

# **Part 3 — Visualization (Matplotlib + Seaborn)**

# 15. Line chart — Monthly total sales trend over the two years.

In [ ]:
df.head(1)

In [ ]:
df['Year'] = df['Year'].astype(str)
plt.figure(figsize=(10,6))
sns.lineplot(data=df, x='Month',y='Sales',hue='Year',errorbar=None)
plt.show()

In [ ]:
monthly_sales = df.groupby('Month')['Sales'].sum()

plt.figure(figsize=(10,5))
plt.plot(monthly_sales.index, monthly_sales.values, marker='o')
plt.title('Monthly Sales Trend')
plt.xlabel('Month')
plt.ylabel('Total Sales')
plt.xticks(range(1,13))
plt.grid(True)
plt.show()

# 16. Bar chart — Total revenue by Category.

In [ ]:
df.head(1)

In [ ]:
cat_sale = df.groupby('Category')['Sales'].sum().reset_index()
cat_sale

In [ ]:
plt.figure(figsize=(12,8))
sns.barplot(data=cat_sale,x='Category',y='Sales',hue='Category',errorbar=None)
plt.show()

# 17. Boxplot — Sales distribution by Category (this will show your outliers clearly).

In [ ]:
plt.figure(figsize=(10,7))
sns.boxplot(data=df,x='Category',y = 'Sales',hue='Category')
plt.show()

# 18. Histogram / KDE — Distribution of CustomerAge.

In [ ]:
df.head(3)

In [ ]:
sns.histplot(data=df,x='CustomerAge',kde=True)
plt.show()

# 19. Scatter plot — Discount(%) vs Profit, colored by Category

In [ ]:
sns.scatterplot(data=df,x = 'Discount(%)',y='Profit',hue='Category')

# 20. Heatmap — Correlation matrix of numeric columns

In [ ]:
df.info()

In [ ]:
num_col =df[['CustomerAge', 'Profit', 'Quantity', 'Discount(%)', 'ShippingCost', 'Rating']].corr()
sns.heatmap(num_col,annot=True)
plt.show()

# 21. Countplot — Number of orders by PaymentMethod, split by Gender.

In [ ]:
plt.figure(figsize=(10,7))
sns.countplot(data=df,x='PaymentMethod',hue='Gender')
plt.show()

# 22. Pairplot — Relationships between Sales, Profit, Quantity, and Discount(%).

In [ ]:
plt.figure(figsize=(30,10))
re = df[['Sales','Profit']]
sns.pairplot(re)
plt.show()

# **Part 4 — "Client Questions" (Bring it together)**

# 23. If RetailCo could only focus on 3 cities next quarter, which would you recommend and why?